# Pipeline de clima — versão com Facade

O mesmo pipeline do [`01_go_horse.ipynb`](01_go_horse.ipynb), agora como cliente da fachada `PipelineClima`. Os subsistemas (extração, validação, limpeza, agregação, carga e relatório) moram no pacote `clima/`. Este notebook não precisa conhecer nenhum deles.

In [1]:
import sys
sys.path.insert(0, "..")  # para importar o pacote clima/ da pasta do projeto

from clima import PipelineClima

CIDADES = ["Rio de Janeiro", "São Paulo", "Belo Horizonte"]

## Uma chamada, da API ao relatório

In [2]:
pipeline = PipelineClima(pasta_saida="../saida")
resultado = pipeline.executar(CIDADES, "2026-01-01", "2026-03-31")
resultado.mensal

,cidade,mes,tmax_media,tmin_media,amplitude_media,chuva_total_mm,dias_chuvosos
0,Belo Horizonte,2026-01,25.8,18.3,7.5,432.8,22
1,Belo Horizonte,2026-02,26.3,18.8,7.5,378.5,25
2,Belo Horizonte,2026-03,26.9,18.3,8.5,164.1,24
3,Rio de Janeiro,2026-01,30.6,23.4,7.2,142.9,16
4,Rio de Janeiro,2026-02,30.5,24.3,6.2,212.4,20
5,Rio de Janeiro,2026-03,28.6,22.8,5.8,85.6,11
6,São Paulo,2026-01,26.9,18.3,8.6,282.0,27
7,São Paulo,2026-02,27.2,19.4,7.9,280.9,23
8,São Paulo,2026-03,27.5,18.0,9.4,171.1,14


## O mesmo caso novo: CSV com defeitos

Na versão go horse, esse caso virou uma cópia do notebook. Aqui é **só outra fonte**, e o cliente continua com a mesma chamada.

In [3]:
pipeline_csv = PipelineClima.offline("../dados/clima_sujo.csv", pasta_saida="../saida")
resultado_csv = pipeline_csv.executar(CIDADES, "2026-01-01", "2026-03-31")
print(resultado_csv.validacao)
resultado_csv.mensal

RelatorioValidacao(linhas=273, duplicadas=3, nulos={'tmax': 1, 'tmin': 1, 'chuva_mm': 1}, fora_da_faixa=2, tmin_maior_que_tmax=1)


,cidade,mes,tmax_media,tmin_media,amplitude_media,chuva_total_mm,dias_chuvosos
0,Belo Horizonte,2026-01,25.8,18.3,7.5,432.8,22
1,Belo Horizonte,2026-02,26.3,18.8,7.5,378.5,25
2,Belo Horizonte,2026-03,26.9,18.3,8.5,164.1,24
3,Rio de Janeiro,2026-01,30.6,23.4,7.2,142.9,16
4,Rio de Janeiro,2026-02,30.5,24.3,6.2,212.4,20
5,Rio de Janeiro,2026-03,28.6,22.8,5.8,85.6,11
6,São Paulo,2026-01,26.9,18.3,8.6,281.6,27
7,São Paulo,2026-02,27.2,19.4,7.9,280.9,23
8,São Paulo,2026-03,27.5,18.0,9.4,171.0,14


A validação contou os problemas e a limpeza tratou cada um: removeu as duplicatas, descartou os valores impossíveis, destrocou `tmin`/`tmax` e interpolou as lacunas. Rodar de novo não duplica o banco, porque a carga faz upsert pela chave `(cidade, data)`.

## O subsistema continua acessível

A fachada não tranca nada. Quem precisa de controle fino usa as classes diretamente.

In [4]:
from clima.limpeza import Limpador
from clima.validacao import Validador
import pandas as pd

bruto = pd.read_csv("../dados/clima_sujo.csv")
print(Validador(temp_max=40).validar(bruto))
Limpador(temp_max=40).limpar(bruto).head()

RelatorioValidacao(linhas=273, duplicadas=3, nulos={'tmax': 1, 'tmin': 1, 'chuva_mm': 1}, fora_da_faixa=2, tmin_maior_que_tmax=1)


,cidade,data,tmax,tmin,chuva_mm
0,Belo Horizonte,2026-01-01,31.6,21.7,5.6
1,Belo Horizonte,2026-01-02,29.7,20.8,1.7
2,Belo Horizonte,2026-01-03,27.2,20.1,26.2
3,Belo Horizonte,2026-01-04,22.7,18.4,53.1
4,Belo Horizonte,2026-01-05,19.8,17.4,68.8


## O que mudou em relação ao go horse

| Ponto de dor | Com Facade |
| --- | --- |
| Cliente conhece URL, colunas e ordem dos passos | Cliente conhece `executar(cidades, inicio, fim)` |
| Caso novo vira cópia | Caso novo vira outra fonte injetada |
| Trocar a fonte mexe em tudo | Muda só a fachada (ou o parâmetro `fonte`) |
| Dado ruim passa calado | `Validador` mede e `Limpador` trata |
| Estado global entre células | Cada execução devolve um `ResultadoPipeline` |
| Não dá para testar | `tests/` roda o pipeline offline com uma fonte falsa |

**Custo:** mais arquivos e mais código para ler. E se tudo for parar dentro de `PipelineClima`, ela vira um *god object*; a saída é dividir em fachadas menores.